In [2]:
%pip install funasr modelscope transformers sentencepiece openai torchaudio openpyxl pyannote.audio

Note: you may need to restart the kernel to use updated packages.


In [ ]:
% 

In [1]:
import torch 
print(f'CUDA Available: {torch.cuda.is_available()}'); print(f'GPU: {torch.cuda.get_device_name(0)}')

CUDA Available: False


AssertionError: Torch not compiled with CUDA enabled

# Installation & Setup

In [ ]:

import torch
import torchaudio
import os
import gc
import pandas as pd
from tqdm import tqdm
from funasr import AutoModel
from pyannote.audio import Pipeline
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText
from openai import OpenAI
from tkinter import Tk, filedialog

# --- Configuration ---
HF_TOKEN = "YOUR_HUGGING_FACE_TOKEN"
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Running on: {DEVICE}")

# Model Initialization

In [ ]:
print("Loading Transcription & Diarization Models...")
# 1. SenseVoice for Mandarin + Emotions
transcription_model = AutoModel(model="iic/SenseVoiceSmall", device=DEVICE, hub="ms")

# 2. Pyannote for Speaker ID
diarize_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=HF_TOKEN).to(torch.device(DEVICE))

# 3. SeamlessM4T for Direct Translation
processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
direct_model = SeamlessM4Tv2ForSpeechToText.from_pretrained("facebook/seamless-m4t-v2-large").to(DEVICE)

# 4. OpenAI for Chained Translation
client = OpenAI(api_key=OPENAI_API_KEY)

# Processing Functions

In [ ]:
def get_chained_translation(zh_text):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a research assistant. Translate this Mandarin transcript to English. Preserve all emotional tags like <|HAPPY|> or <|LAUGHTER|>."},
            {"role": "user", "content": zh_text}
        ]
    )
    return response.choices[0].message.content

def get_direct_translation(audio_path):
    audio, orig_freq = torchaudio.load(audio_path)
    if orig_freq != 16000:
        audio = torchaudio.functional.resample(audio, orig_freq=orig_freq, new_freq=16000)
    inputs = processor(audios=audio, return_tensors="pt").to(DEVICE)
    output_tokens = direct_model.generate(**inputs, tgt_lang="eng")
    return processor.decode(output_tokens[0].tolist()[0], skip_special_tokens=True)

# Processing Functions

In [ ]:
def get_chained_translation(zh_text):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a research assistant. Translate this Mandarin transcript to English. Preserve all emotional tags like <|HAPPY|> or <|LAUGHTER|>."},
            {"role": "user", "content": zh_text}
        ]
    )
    return response.choices[0].message.content

def get_direct_translation(audio_path):
    audio, orig_freq = torchaudio.load(audio_path)
    if orig_freq != 16000:
        audio = torchaudio.functional.resample(audio, orig_freq=orig_freq, new_freq=16000)
    inputs = processor(audios=audio, return_tensors="pt").to(DEVICE)
    output_tokens = direct_model.generate(**inputs, tgt_lang="eng")
    return processor.decode(output_tokens[0].tolist()[0], skip_special_tokens=True)

# The Batch Evaluator Execution

In [ ]:
def run_full_evaluation():
    root = Tk(); root.withdraw(); root.attributes('-topmost', True)
    input_dir = filedialog.askdirectory(title="Select Source Audio Folder")
    if not input_dir: return

    results = []
    extensions = ('.wav', '.mp3', '.m4a', '.mp4')
    files = [os.path.join(r, f) for r, _, fs in os.walk(input_dir) for f in fs if f.lower().endswith(extensions)]

    for audio_path in tqdm(files, desc="Processing Research Data"):
        try:
            # A. Transcribe (SenseVoice)
            res = transcription_model.generate(input=audio_path, language="zh", use_itn=True)
            zh_text = res[0]['text']
            
            # B. Diarize (Pyannote)
            diarization = diarize_pipeline(audio_path)
            speakers = ", ".join(list(set([label for _, _, label in diarization.itertracks(yield_label=True)])))

            # C. Translate (Both Methods)
            chained_en = get_chained_translation(zh_text)
            direct_en = get_direct_translation(audio_path)

            results.append({
                "File": os.path.basename(audio_path),
                "Speakers": speakers,
                "Original Mandarin": zh_text,
                "Chained English (Preserves Tags)": chained_en,
                "Direct English (Seamless)": direct_en
            })
            
            # Memory Cleanup
            torch.cuda.empty_cache(); gc.collect()
        except Exception as e:
            print(f"Error on {audio_path}: {e}")

    # Export
    df = pd.DataFrame(results)
    df.to_excel("NGSS_Translation_Evaluation.xlsx", index=False)
    print("Batch Complete. Check 'NGSS_Translation_Evaluation.xlsx'")

run_full_evaluation()

# Visualization & Final Export

import matplotlib.pyplot as plt

def visualize_and_finalize(excel_path="NGSS_Translation_Evaluation.xlsx"):
    # 1. Load the results we just created
    df = pd.read_excel(excel_path)
    
    # 2. Calculate word counts for comparison
    df['Chained_Word_Count'] = df['Chained English (Preserves Tags)'].apply(lambda x: len(str(x).split()))
    df['Direct_Word_Count'] = df['Direct English (Seamless)'].apply(lambda x: len(str(x).split()))
    
    # 3. Create the Visualization
    plt.figure(figsize=(12, 6))
    x = range(len(df))
    width = 0.35
    
    plt.bar(x, df['Chained_Word_Count'], width, label='Chained (SenseVoice+GPT)', color='#4A90E2')
    plt.bar([i + width for i in x], df['Direct_Word_Count'], width, label='Direct (SeamlessM4T)', color='#50E3C2')
    
    plt.xlabel('Audio Files (Index)')
    plt.ylabel('Word Count')
    plt.title('Translation Verbosity Comparison: Chained vs. Direct')
    plt.xticks([i + width/2 for i in x], df.index)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Save the chart for your research folder
    plt.savefig("Translation_Comparison_Chart.png")
    plt.show()
    
    print(f"Visual analysis complete. Summary table:\n")
    print(df[['File', 'Chained_Word_Count', 'Direct_Word_Count']].describe())

# Run the visualization
# visualize_and_finalize()